In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

silver_table = "crypto_project.crypto.silver_crypto_market"
gold_dim_coins = "crypto_project.crypto.dim_coins"
gold_fact_daily = "crypto_project.crypto.fact_crypto_daily"

# Read Silver and filter strictly to Top 200 coins by market cap rank
df_silver_top200 = (
    spark.table(silver_table)
    .filter(F.col("market_cap_rank").isNotNull() & (F.col("market_cap_rank") <= 200))
)

# 1. Dimension Table: MERGE latest coin metadata for Top 200 universe
latest_coin_win = Window.partitionBy("coin_id").orderBy(F.col("market_date").desc())
df_dim_updates = (
    df_silver_top200
    .withColumn("rn", F.row_number().over(latest_coin_win))
    .filter(F.col("rn") == 1)
    .select("coin_id", "symbol", "name", "market_cap_rank")
)

if not spark.catalog.tableExists(gold_dim_coins):
    df_dim_updates.write.format("delta").saveAsTable(gold_dim_coins)
else:
    DeltaTable.forName(spark, gold_dim_coins).alias("t").merge(
        df_dim_updates.alias("s"),
        "t.coin_id = s.coin_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# 2. Fact Table: Compute rolling 7-day windows on a 10-day lookback, then MERGE
df_recent_top200 = df_silver_top200.filter(
    F.col("market_date") >= F.date_sub(F.current_date(), 10)
)

rolling_7d = Window.partitionBy("coin_id").orderBy("market_date").rowsBetween(-6, 0)
prev_day = Window.partitionBy("coin_id").orderBy("market_date")

df_fact_updates = (
    df_recent_top200
    .withColumn("prev_price_usd", F.lag("price_usd", 1).over(prev_day))
    .withColumn("sma_7d_price", F.round(F.avg("price_usd").over(rolling_7d), 4))
    .withColumn("volatility_7d", F.round(F.stddev("price_usd").over(rolling_7d), 4))
    .withColumn("volume_to_mcap_ratio", F.round(F.try_divide(F.col("total_volume"), F.col("market_cap")), 4))
    .withColumn(
        "trend_signal",
        F.when(F.col("price_usd") > F.col("sma_7d_price"), "ABOVE_7D_SMA")
         .when(F.col("price_usd") < F.col("sma_7d_price"), "BELOW_7D_SMA")
         .otherwise("NEUTRAL")
    )
    .select(
        "coin_id", "symbol", "market_date", "price_usd", "prev_price_usd",
        "price_change_24h_pct", "market_cap", "total_volume",
        "sma_7d_price", "volatility_7d", "volume_to_mcap_ratio", "trend_signal"
    )
)

if not spark.catalog.tableExists(gold_fact_daily):
    df_fact_updates.write.format("delta").saveAsTable(gold_fact_daily)
else:
    DeltaTable.forName(spark, gold_fact_daily).alias("t").merge(
        df_fact_updates.alias("s"),
        """
        t.coin_id = s.coin_id
        AND t.market_date = s.market_date
        AND t.market_date >= date_sub(current_date(), 10)
        """
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print("Gold Dim & Fact incrementally merged for Top 200 coins—all past historical rows preserved!")

Gold Dim & Fact incrementally merged for Top 200 coins—all past historical rows preserved!
